In [34]:
import pandas as pd
import sqlite3
from pathlib import Path

In [35]:
# Project directories

BASE_DIR = Path.cwd().parent

PROCESSED_DATA_PATH = (
    BASE_DIR
    / "Data"
    / "Processed"
    / "diabetic_data_processed.csv"
)

DATABASE_DIR = (
    BASE_DIR
    / "Database"
)

DATABASE_PATH = (
    DATABASE_DIR
    / "healthsense_ai.db"
)

print("Project directory:")
print(BASE_DIR)

print("\nProcessed data path:")
print(PROCESSED_DATA_PATH)

print("\nDatabase path:")
print(DATABASE_PATH)


Project directory:
c:\Users\Sai\Documents\HealthSense_AI

Processed data path:
c:\Users\Sai\Documents\HealthSense_AI\Data\Processed\diabetic_data_processed.csv

Database path:
c:\Users\Sai\Documents\HealthSense_AI\Database\healthsense_ai.db


In [36]:
df = pd.read_csv(
    PROCESSED_DATA_PATH
)

print("Dataset loaded successfully.")

print(
    f"Rows: {df.shape[0]}"
)

print(
    f"Columns: {df.shape[1]}"
)

Dataset loaded successfully.
Rows: 101766
Columns: 50


In [37]:
df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,max_glu_serum_available,A1Cresult_available
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,0,0,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,...,No,No,No,No,No,Ch,Yes,1,0,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,...,No,No,No,No,No,No,Yes,0,0,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,...,No,No,No,No,No,Ch,Yes,0,0,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,...,No,No,No,No,No,Ch,Yes,0,0,0


In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   admission_type_id         101766 non-null  int64
 6   discharge_disposition_id  101766 non-null  int64
 7   admission_source_id       101766 non-null  int64
 8   time_in_hospital          101766 non-null  int64
 9   medical_specialty         101766 non-null  str  
 10  num_lab_procedures        101766 non-null  int64
 11  num_procedures            101766 non-null  int64
 12  num_medications           101766 non-null  int64
 13  number_outpatient         101766 non-null  int64
 14  number_emergency          10176

In [39]:
df.isna().sum().sum()

np.int64(0)

In [40]:
df.duplicated().sum()

np.int64(0)

In [41]:
connection = sqlite3.connect(
    DATABASE_PATH
)

print(
    "SQLite database connection successful."
)

SQLite database connection successful.


In [42]:
df.to_sql(
    "patient_encounters",
    connection,
    if_exists="replace",
    index=False
)

print(
    "patient_encounters table created successfully."
)

patient_encounters table created successfully.


In [43]:
query = """
SELECT COUNT(*) AS total_records
FROM patient_encounters;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,total_records
0,101766


In [44]:
query = """
SELECT COUNT(DISTINCT patient_nbr) AS unique_patients
FROM patient_encounters;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,unique_patients
0,71518


In [45]:
query = """
SELECT
    readmitted,
    COUNT(*) AS encounter_count
FROM patient_encounters
GROUP BY readmitted
ORDER BY readmitted;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,readmitted,encounter_count
0,0,54864
1,1,46902


In [46]:
query = """
SELECT
    readmitted,
    COUNT(*) AS encounter_count,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM patient_encounters),
        2
    ) AS percentage
FROM patient_encounters
GROUP BY readmitted;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,readmitted,encounter_count,percentage
0,0,54864,53.91
1,1,46902,46.09


In [47]:
query = """
SELECT
    ROUND(
        AVG(time_in_hospital),
        2
    ) AS average_hospital_stay
FROM patient_encounters;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,average_hospital_stay
0,4.4


In [48]:
query = """
SELECT
    age,
    COUNT(*) AS total_encounters,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY age
ORDER BY readmission_rate DESC;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,age,total_encounters,readmissions,readmission_rate
0,[80-90),17197,8301,48.27
1,[70-80),26068,12544,48.12
2,[60-70),22483,10399,46.25
3,[20-30),1657,746,45.02
4,[40-50),9685,4305,44.45
5,[50-60),17256,7585,43.96
6,[30-40),3775,1611,42.68
7,[90-100),2793,1118,40.03
8,[10-20),691,264,38.21
9,[0-10),161,29,18.01


In [49]:
query = """
SELECT
    gender,
    COUNT(*) AS total_encounters,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY gender
ORDER BY readmission_rate DESC;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,gender,total_encounters,readmissions,readmission_rate
0,Female,54708,25670,46.92
1,Male,47055,21232,45.12
2,Unknown/Invalid,3,0,0.00


In [50]:
query = """
SELECT
    number_inpatient,
    COUNT(*) AS total_encounters,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY number_inpatient
ORDER BY number_inpatient;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,number_inpatient,total_encounters,readmissions,readmission_rate
0,0,67630,26034,38.49
1,1,19521,10690,54.76
2,2,7566,4913,64.94
3,3,3411,2378,69.72
4,4,1622,1204,74.23
5,5,812,650,80.05
6,6,480,404,84.17
7,7,268,217,80.97
8,8,151,137,90.73
9,9,111,98,88.29


In [51]:
query = """
SELECT
    diag_1,
    COUNT(*) AS encounter_count,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY diag_1
ORDER BY encounter_count DESC
LIMIT 20;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,diag_1,encounter_count,readmissions,readmission_rate
0,428,6862,4057,59.12
1,414,6581,2720,41.33
2,786,4016,1709,42.55
3,410,3614,1438,39.79
4,486,3508,1683,47.98
5,427,2766,1226,44.32
6,491,2275,1360,59.78
7,715,2151,783,36.40
8,682,2042,957,46.87
9,434,2028,917,45.22


In [52]:
query = """
SELECT
    diag_2,
    COUNT(*) AS encounter_count
FROM patient_encounters
GROUP BY diag_2
ORDER BY encounter_count DESC
LIMIT 20;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,diag_2,encounter_count
0,276,6752
1,428,6662
2,250,6071
3,427,5036
4,401,3736
5,496,3305
6,599,3288
7,403,2823
8,414,2650
9,411,2566


In [53]:
query = """
SELECT
    number_emergency,
    COUNT(*) AS total_encounters,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY number_emergency
ORDER BY number_emergency;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,number_emergency,total_encounters,readmissions,readmission_rate
0,0,90383,39673,43.89
1,1,7677,4552,59.29
2,2,2042,1370,67.09
3,3,725,521,71.86
4,4,374,305,81.55
5,5,192,154,80.21
6,6,94,79,84.04
7,7,73,66,90.41
8,8,50,41,82.00
9,9,33,30,90.91


In [54]:
query = """
SELECT
    number_outpatient,
    COUNT(*) AS total_encounters,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY number_outpatient
ORDER BY number_outpatient;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,number_outpatient,total_encounters,readmissions,readmission_rate
0,0,85027,37115,43.65
1,1,8547,4904,57.38
2,2,3594,2160,60.10
3,3,2042,1173,57.44
4,4,1099,646,58.78
5,5,533,306,57.41
6,6,303,194,64.03
7,7,155,107,69.03
8,8,98,45,45.92
9,9,83,51,61.45


In [55]:
query = """
SELECT
    time_in_hospital,
    COUNT(*) AS total_encounters,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY time_in_hospital
ORDER BY time_in_hospital;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,time_in_hospital,total_encounters,readmissions,readmission_rate
0,1,14208,5633,39.65
1,2,17224,7631,44.30
2,3,17756,7979,44.94
3,4,13924,6708,48.18
4,5,9966,4785,48.01
5,6,7539,3704,49.13
6,7,5859,2875,49.07
7,8,4391,2214,50.42
8,9,3002,1511,50.33
9,10,2342,1174,50.13


In [56]:
query = """
SELECT
    number_diagnoses,
    COUNT(*) AS total_encounters,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY number_diagnoses
ORDER BY number_diagnoses;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,number_diagnoses,total_encounters,readmissions,readmission_rate
0,1,219,52,23.74
1,2,1023,336,32.84
2,3,2835,972,34.29
3,4,5537,2063,37.26
4,5,11393,4039,35.45
5,6,10161,4455,43.84
6,7,10393,4851,46.68
7,8,10616,5052,47.59
8,9,49474,25026,50.58
9,10,17,8,47.06


In [57]:
query = """
SELECT
    num_medications,
    COUNT(*) AS total_encounters,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY num_medications
ORDER BY num_medications;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,num_medications,total_encounters,readmissions,readmission_rate
0,1,262,83,31.68
1,2,470,145,30.85
2,3,900,274,30.44
3,4,1417,495,34.93
4,5,2017,687,34.06
...,...,...,...,...
70,72,3,3,100.00
71,74,1,0,0.00
72,75,2,0,0.00
73,79,1,0,0.00


In [58]:
query = """
SELECT
    diabetesMed,
    COUNT(*) AS encounter_count,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY diabetesMed;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,diabetesMed,encounter_count,readmissions,readmission_rate
0,No,23403,9473,40.48
1,Yes,78363,37429,47.76


In [59]:
query = """
SELECT
    insulin,
    COUNT(*) AS encounter_count,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY insulin
ORDER BY encounter_count DESC;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,insulin,encounter_count,readmissions,readmission_rate
0,No,47383,20705,43.70
1,Steady,30849,13915,45.11
2,Down,12218,6450,52.79
3,Up,11316,5832,51.54


In [60]:
query = """
SELECT
    max_glu_serum_available,
    COUNT(*) AS encounter_count,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY max_glu_serum_available;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,max_glu_serum_available,encounter_count,readmissions,readmission_rate
0,0,96420,44305,45.95
1,1,5346,2597,48.58


In [61]:
query = """
SELECT
    A1Cresult_available,
    COUNT(*) AS encounter_count,
    SUM(readmitted) AS readmissions,
    ROUND(
        SUM(readmitted) * 100.0 /
        COUNT(*),
        2
    ) AS readmission_rate
FROM patient_encounters
GROUP BY A1Cresult_available;
"""

result = pd.read_sql_query(
    query,
    connection
)

result

,A1Cresult_available,encounter_count,readmissions,readmission_rate
0,0,84748,39426,46.52
1,1,17018,7476,43.93
